In [28]:
import pandas as pd
from pathlib import Path

processed_path = Path(
    r"D:\MScProjects\labour-market-intelligence\data\processed"
)

skills = pd.read_csv(
    processed_path / "onet_selected_skills.csv"
)

print("Shape:", skills.shape)
print("\nColumns:")
print(skills.columns.tolist())

skills.head(10)

Shape: (80, 4)

Columns:
['Title', 'Element Name', 'Scale Name', 'Data Value']


,Title,Element Name,Scale Name,Data Value
0,Information Security Analysts,Reading Comprehension,Importance,4.00
1,Information Security Analysts,Reading Comprehension,Level,4.12
2,Information Security Analysts,Active Listening,Importance,3.75
3,Information Security Analysts,Active Listening,Level,3.88
4,Information Security Analysts,Writing,Importance,3.50
5,Information Security Analysts,Writing,Level,3.75
6,Information Security Analysts,Speaking,Importance,3.62
7,Information Security Analysts,Speaking,Level,3.75
8,Information Security Analysts,Mathematics,Importance,2.25
9,Information Security Analysts,Mathematics,Level,2.38


In [29]:
print(skills["Title"].unique())
print("\nNumber of occupations:", skills["Title"].nunique())

print("\nSkills:")
print(skills["Element Name"].unique())

<ArrowStringArray>
['Information Security Analysts',       'Database Administrators',
           'Software Developers',                 'Statisticians']
Length: 4, dtype: str

Number of occupations: 4

Skills:
<ArrowStringArray>
['Reading Comprehension',      'Active Listening',               'Writing',
              'Speaking',           'Mathematics',               'Science',
     'Critical Thinking',       'Active Learning',   'Learning Strategies',
            'Monitoring']
Length: 10, dtype: str


In [30]:
onet_path = Path(
    r"D:\MScProjects\labour-market-intelligence\data\raw\onet"
)

software = pd.read_excel(
    onet_path / "Software Skills.xlsx"
)

print("Shape:", software.shape)

print("\nColumns:")
print(software.columns.tolist())

software.head(10)

Shape: (31821, 7)

Columns:
['O*NET-SOC Code', 'Title', 'Workplace Example', 'Element ID', 'Element Name', 'Hot Technology', 'In Demand']


,O*NET-SOC Code,Title,Workplace Example,Element ID,Element Name,Hot Technology,In Demand
0,11-1011.00,Chief Executives,Adobe Acrobat,2.E.5.b,Document management software,Y,N
1,11-1011.00,Chief Executives,AdSense Tracker,2.E.6.f,Data base user interface and query software,N,N
2,11-1011.00,Chief Executives,Atlassian JIRA,2.E.5.a,Content workflow software,Y,N
3,11-1011.00,Chief Executives,Blackbaud The Raiser's Edge,2.E.6.c,Customer relationship management CRM software,N,N
4,11-1011.00,Chief Executives,ComputerEase construction accounting software,2.E.2.a,Accounting software,N,N
5,11-1011.00,Chief Executives,Database reporting software,2.E.6.e,Data base reporting software,N,N
6,11-1011.00,Chief Executives,Databox,2.E.6.f,Data base user interface and query software,N,N
7,11-1011.00,Chief Executives,Email software,2.E.16.a,Electronic mail software,N,N
8,11-1011.00,Chief Executives,Exact Software Macola ES Labor Performance,2.E.2.e,Time accounting software,N,N
9,11-1011.00,Chief Executives,Extensible markup language XML,2.E.7.c,Enterprise application integration software,Y,N


In [31]:
keywords = (
    r"artificial intelligence|machine learning|tensorflow|pytorch|"
    r"cyber|security|firewall|penetration|siem|"
    r"python|sql|tableau|power bi|r programming|data analytics|data analysis"
)

matches = software[
    software["Workplace Example"]
    .astype(str)
    .str.contains(keywords, case=False, na=False)
]

print("Matching rows:", len(matches))

print("\nMatching technologies:")
print(
    matches["Workplace Example"]
    .drop_duplicates()
    .sort_values()
    .to_list()
)

Matching rows: 907

Matching technologies:
['2AB iLock Security Services', 'AdaptaSoft CyberPay', 'Artificial intelligence software', 'Aspyra CyberLAB', 'Aspyra CyberPATH', 'Bing for Power BI', 'CAST SQL Builder', 'COATSsql', 'Cyber Records MediChart Express', 'CyberArk', 'CyberDyne Industries CaseWizard', 'CyberMatrix', 'CyberMatrix POS', 'CyberShift Workforce Management 3G Time and Attendance', 'CyberSoft NutriBase', 'Cybermetrics GAGETrak', 'Cybermotion 3D Designer', 'Cybersoft Primero Software Suite', 'Data Recovery Software SQL Server Data Recovery', 'Data analysis software', 'Database security software', 'EpiData Analysis', 'European Southern Observatory Munich Image Data Analysis System ESO-MIDAS', 'Firewall software', 'IBM QRadar SIEM', 'IBM Security Network Intrusion Prevention System', 'Infoblox security and networking software', 'Internet Protocol Security IPSEC', 'Juniper Networks NetScreen-Security Manager', 'McAfee Enterprise Security Manager', 'Micosoft SQL Server Analys

In [32]:
import re

def classify_skill(x):
    x = str(x).lower().strip()

    # AI
    if any(k in x for k in [
        "artificial intelligence",
        "machine learning",
        "tensorflow",
        "pytorch"
    ]):
        return "AI"

    # Cybersecurity
    if any(k in x for k in [
        "cyberark",
        "cybersecurity",
        "database security",
        "firewall",
        "qradar siem",
        "intrusion prevention",
        "network security",
        "ipsec",
        "security manager",
        "enterprise security manager",
        "penetration testing",
        "security assertion",
        "security incident",
        "security risk assessment",
        "security testing"
    ]):
        return "Cybersecurity"

    # Data Analytics
    if any(k in x for k in [
        "data analysis",
        "power bi",
        "tableau",
        "python",
        "structured query language sql",
        "structure query language sql",
        "transact-sql",
        "t-sql",
        "mysql",
        "postgresql",
        "sqlite",
        "pl/sql",
        "sql server",
        "sql developer",
        "nosql"
    ]):
        return "Data Analytics"

    return None


software["Skill_Category"] = (
    software["Workplace Example"]
    .apply(classify_skill)
)

classified = software[
    software["Skill_Category"].notna()
].copy()

print(classified["Skill_Category"].value_counts())

print("\nUnique technologies by category:")
for category in ["AI", "Cybersecurity", "Data Analytics"]:
    print(f"\n--- {category} ---")
    print(
        sorted(
            classified.loc[
                classified["Skill_Category"] == category,
                "Workplace Example"
            ].unique()
        )
    )

Skill_Category
Data Analytics    728
Cybersecurity      97
AI                 10
Name: count, dtype: int64

Unique technologies by category:

--- AI ---
['Artificial intelligence software', 'PyTorch', 'TensorFlow']

--- Cybersecurity ---
['CyberArk', 'Database security software', 'Firewall software', 'IBM QRadar SIEM', 'IBM Security Network Intrusion Prevention System', 'Internet Protocol Security IPSEC', 'Intrusion prevention system IPS', 'Juniper Networks NetScreen-Security Manager', 'McAfee Enterprise Security Manager', 'Network intrusion prevention systems NIPS', 'Network security auditing software', 'NortonLifeLock cybersecurity software', 'Penetration testing software', 'Security assertion markup language SAML', 'Security incident management software', 'Security risk assessment software', 'Security testing software']

--- Data Analytics ---
['Bing for Power BI', 'Data Recovery Software SQL Server Data Recovery', 'Data analysis software', 'EpiData Analysis', 'European Southern Obs

In [33]:
occupation_skill_summary = (
    classified
    .groupby(["Skill_Category", "Title"])
    .agg(
        Technology_Count=("Workplace Example", "nunique"),
        Hot_Technology_Count=("Hot Technology", lambda x: (x == "Y").sum()),
        In_Demand_Count=("In Demand", lambda x: (x == "Y").sum())
    )
    .reset_index()
)

# Show top 10 occupations in each category
for category in ["AI", "Cybersecurity", "Data Analytics"]:
    print(f"\n--- {category} ---")

    display(
        occupation_skill_summary[
            occupation_skill_summary["Skill_Category"] == category
        ]
        .sort_values(
            "Technology_Count",
            ascending=False
        )
        .head(10)
    )


--- AI ---


,Skill_Category,Title,Technology_Count,Hot_Technology_Count,In_Demand_Count
0,AI,Computer and Information Research Scientists,2,2,2
1,AI,Data Scientists,2,2,2
2,AI,Financial Risk Specialists,2,2,0
5,AI,Software Developers,2,2,0
3,AI,Industrial Engineering Technologists and Techn...,1,0,0
4,AI,Intelligence Analysts,1,1,0



--- Cybersecurity ---


,Skill_Category,Title,Technology_Count,Hot_Technology_Count,In_Demand_Count
24,Cybersecurity,Information Security Analysts,10,0,1
8,Cybersecurity,Computer Network Architects,7,0,1
9,Cybersecurity,Computer Network Support Specialists,5,0,1
32,Cybersecurity,Network and Computer Systems Administrators,5,0,1
42,Cybersecurity,Security Management Specialists,5,0,1
25,Cybersecurity,Information Security Engineers,4,0,1
36,Cybersecurity,Sales Engineers,3,0,0
45,Cybersecurity,Software Quality Assurance Analysts and Testers,3,0,0
33,Cybersecurity,Penetration Testers,3,0,1
20,Cybersecurity,Digital Forensics Analysts,3,0,1



--- Data Analytics ---


,Skill_Category,Title,Technology_Count,Hot_Technology_Count,In_Demand_Count
112,Data Analytics,Database Architects,16,13,4
111,Data Analytics,Database Administrators,15,13,11
241,Data Analytics,Software Developers,15,13,4
77,Data Analytics,Business Intelligence Analysts,13,13,4
242,Data Analytics,Software Quality Assurance Analysts and Testers,13,11,2
97,Data Analytics,Computer Systems Analysts,13,13,3
110,Data Analytics,Data Warehousing Specialists,12,12,4
182,Data Analytics,Management Analysts,12,12,4
169,Data Analytics,Information Technology Project Managers,11,11,1
201,Data Analytics,Network and Computer Systems Administrators,11,11,2


In [34]:
output_path = processed_path / "digital_skill_classification.csv"

classified[
    [
        "O*NET-SOC Code",
        "Title",
        "Workplace Example",
        "Element Name",
        "Hot Technology",
        "In Demand",
        "Skill_Category"
    ]
].to_csv(output_path, index=False)

print("Saved:", output_path)
print("Shape:", classified.shape)

print("\nCategory counts:")
print(classified["Skill_Category"].value_counts())

Saved: D:\MScProjects\labour-market-intelligence\data\processed\digital_skill_classification.csv
Shape: (835, 8)

Category counts:
Skill_Category
Data Analytics    728
Cybersecurity      97
AI                 10
Name: count, dtype: int64


In [35]:
from pathlib import Path

raw_path = Path(
    r"D:\MScProjects\labour-market-intelligence\data\raw"
)

# Look for CSV/Excel files that may contain vacancy/job text
for p in raw_path.rglob("*"):
    if p.suffix.lower() in [".csv", ".xlsx", ".xls"]:
        print(p)

D:\MScProjects\labour-market-intelligence\data\raw\onet\Essential Skills.xlsx
D:\MScProjects\labour-market-intelligence\data\raw\onet\Job Titles.xlsx
D:\MScProjects\labour-market-intelligence\data\raw\onet\Occupation Data.xlsx
D:\MScProjects\labour-market-intelligence\data\raw\onet\Software Skills.xlsx
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_Online_Job_Advert_Salaries_2017_2025.xlsx
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_Textkernel_Job_Adverts.xlsx
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Occupation_2023\Occupation SOC20 (4) Table 14.10a   Paid hours worked - Basic 2023.xlsx
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Occupation_2023\Occupation SOC20 (4) Table 14.10b   Paid hours worked - Basic 2023 CV.xlsx
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Occupation_2023\Occupation SOC20 (4) Table 14.11a   Paid hours worked - Overtime 2023.xlsx
D:\MScProjects\labour-market-intelligence\data

In [36]:
textkernel_path = (
    raw_path / "ons" / "ONS_Textkernel_Job_Adverts.xlsx"
)

tk = pd.read_excel(textkernel_path)

print("Shape:", tk.shape)

print("\nColumns:")
print(tk.columns.tolist())

tk.head(10)

Shape: (24, 1)

Columns:
['New Online Job Adverts']


,New Online Job Adverts
0,This workbook contains 3 data tables
1,Source: Textkernel
2,Data coverage: Jan 2018 - Jun 2026
3,Date of publication: 23 July 2026
4,Date of next publication: to be confirmed
5,This dataset is part of 'Economic activity and...
6,Latest release (opens in new browser window)
7,Latest dashboard (opens in new browser window)
8,View the contents of this dataset
9,Contact details


In [37]:
xls = pd.ExcelFile(textkernel_path)

print(xls.sheet_names)

['Cover', 'Contents', 'Notes', '1.Region NSA', '2.Occupation 2-Digit NSA', '3.Occupation 4-Digit NSA']


In [38]:
tk_occ = pd.read_excel(
    textkernel_path,
    sheet_name="3.Occupation 4-Digit NSA",
    header=None
)

print("Shape:", tk_occ.shape)

tk_occ.head(20)

Shape: (107, 415)


,0,1,2,3,4,5,6,7,8,9,...,405,406,407,408,409,410,411,412,413,414
0,"Table 3: New online job adverts in the UK, spl...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,This worksheet contains one table. See Notes w...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Back to Notes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Back to Contents,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Month,All Occupations,1111 Chief executives and senior officials,1112 Elected officers and representatives,1121 Production managers and directors in manu...,1122 Production managers and directors in cons...,1123 Production managers and directors in mini...,1131 Financial managers and directors,"1132 Marketing, sales and advertising directors",1133 Public relations and communications direc...,...,9259 Elementary storage occupations n.e.c.,9261 Bar and catering supervisors,9262 Hospital porters,9263 Kitchen and catering assistants,9264 Waiters and waitresses,9265 Bar staff,9266 Coffee shop workers,9267 Leisure and theme park attendants,9269 Other elementary services occupations n.e.c.,Unknown
5,2018-01-01 00:00:00,993156,944,643,5355,3893,151,9622,3887,475,...,703,1559,79,9633,3096,2867,635,1389,828,1248
6,2018-02-01 00:00:00,854140,811,504,4420,3080,259,8053,3242,386,...,654,1293,59,11202,2847,2775,420,1516,880,1090
7,2018-03-01 00:00:00,921648,732,632,5121,3715,173,8532,3807,435,...,701,1571,65,12407,3253,3246,497,1463,930,1041
8,2018-04-01 00:00:00,819751,651,630,4355,3014,242,7502,2993,388,...,618,1333,48,8773,2851,2885,463,2139,748,1007
9,2018-05-01 00:00:00,891434,679,1330,4570,3552,151,8155,3327,393,...,611,1495,52,9488,2934,2749,417,1768,724,1168


In [39]:
job_titles = pd.read_excel(
    raw_path / "onet" / "Job Titles.xlsx"
)

print("Shape:", job_titles.shape)

print("\nColumns:")
print(job_titles.columns.tolist())

job_titles.head(10)

Shape: (57543, 5)

Columns:
['O*NET-SOC Code', 'Title', 'Job Title', 'Short Title', 'Source(s)']


,O*NET-SOC Code,Title,Job Title,Short Title,Source(s)
0,11-1011.00,Chief Executives,Aeronautics Commission Director,NaN,08
1,11-1011.00,Chief Executives,Agency Owner,NaN,10
2,11-1011.00,Chief Executives,Agricultural Services Director,NaN,08
3,11-1011.00,Chief Executives,Arts and Humanities Council Director,NaN,08
4,11-1011.00,Chief Executives,Bank President,NaN,09
5,11-1011.00,Chief Executives,Bureau Chief,NaN,"04,06"
6,11-1011.00,Chief Executives,Business Development Executive (BD Executive),BD Executive,09
7,11-1011.00,Chief Executives,Business Development Officer (BD Officer),BD Officer,09
8,11-1011.00,Chief Executives,Business Enterprise Officer,NaN,08
9,11-1011.00,Chief Executives,Business Executive,NaN,09


In [40]:
print("Shape:", job_titles.shape)
print("Unique O*NET occupations:", job_titles["O*NET-SOC Code"].nunique())
print("Unique job titles:", job_titles["Job Title"].nunique())

print("\nMissing Job Titles:")
print(job_titles["Job Title"].isna().sum())

Shape: (57543, 5)
Unique O*NET occupations: 1016
Unique job titles: 46687

Missing Job Titles:
0


In [41]:
import re

def clean_job_title(text):
    text = str(text).lower()
    
    # Remove text inside brackets
    text = re.sub(r"\([^)]*\)", " ", text)
    
    # Keep letters/numbers/spaces
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    
    # Remove repeated spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


job_titles["Clean_Job_Title"] = (
    job_titles["Job Title"]
    .apply(clean_job_title)
)

print(
    job_titles[
        ["Job Title", "Clean_Job_Title"]
    ].head(20)
)

print(
    "\nUnique cleaned titles:",
    job_titles["Clean_Job_Title"].nunique()
)

                                        Job Title  \
0                 Aeronautics Commission Director   
1                                    Agency Owner   
2                  Agricultural Services Director   
3            Arts and Humanities Council Director   
4                                  Bank President   
5                                    Bureau Chief   
6   Business Development Executive (BD Executive)   
7       Business Development Officer (BD Officer)   
8                     Business Enterprise Officer   
9                              Business Executive   
10                  CEO (Chief Executive Officer)   
11             Chief Administrative Officer (CAO)   
12                  Chief Diversity Officer (CDO)   
13                  Chief Financial Officer (CFO)   
14                Chief Information Officer (CIO)   
15      Chief Information Security Officer (CISO)   
16                Chief Innovation Officer (CINO)   
17                    Chief Nursing Officer (C

In [42]:
title_patterns = {
    "AI": [
        r"\bartificial intelligence\b",
        r"\bmachine learning\b",
        r"\bdeep learning\b",
        r"\bai engineer\b",
        r"\bai scientist\b",
        r"\bml engineer\b"
    ],

    "Cybersecurity": [
        r"\bcybersecurity\b",
        r"\bcyber security\b",
        r"\binformation security\b",
        r"\bnetwork security\b",
        r"\bsecurity analyst\b",
        r"\bsecurity engineer\b",
        r"\bpenetration tester\b",
        r"\bpenetration testing\b",
        r"\bdigital forensics\b"
    ],

    "Data Analytics": [
        r"\bdata analyst\b",
        r"\bdata analytics\b",
        r"\bdata scientist\b",
        r"\bdata science\b",
        r"\bbusiness intelligence\b",
        r"\bbi analyst\b",
        r"\bdatabase analyst\b",
        r"\bdata engineer\b",
        r"\bdata architect\b",
        r"\bdata warehouse\b"
    ]
}


def classify_job_title(text):
    matches = []

    for category, patterns in title_patterns.items():
        if any(re.search(pattern, text) for pattern in patterns):
            matches.append(category)

    if len(matches) == 1:
        return matches[0]

    if len(matches) > 1:
        return "Multiple"

    return None


job_titles["NLP_Category"] = (
    job_titles["Clean_Job_Title"]
    .apply(classify_job_title)
)

nlp_matches = job_titles[
    job_titles["NLP_Category"].notna()
].copy()

print("Category counts:")
print(nlp_matches["NLP_Category"].value_counts())

for category in ["AI", "Cybersecurity", "Data Analytics", "Multiple"]:
    print(f"\n--- {category} ---")

    examples = (
        nlp_matches.loc[
            nlp_matches["NLP_Category"] == category,
            "Job Title"
        ]
        .drop_duplicates()
        .head(30)
        .tolist()
    )

    print(examples)

Category counts:
NLP_Category
Cybersecurity     116
Data Analytics     67
AI                 11
Multiple            2
Name: count, dtype: int64

--- AI ---
['AI Engineer (Artificial Intelligence Engineer)', 'Artificial Intelligence Specialist (AI Specialist)', 'Machine Learning Engineer', 'Machine Learning Research Scientist', 'Machine Learning Scientist', 'Machine Learning Software Engineer', 'Artificial Intelligence Specialist']

--- Cybersecurity ---
['Chief Information Security Officer (CISO)', 'Cybersecurity All-Source Collection Manager', 'Cybersecurity All-Source Collection Requirements Manager', 'Information Security Manager', 'Cybersecurity Analyst', 'Cybersecurity Associate', 'Cybersecurity Consultant', 'Cybersecurity Risk Analyst', 'Cybersecurity Specialist', 'Industrial Control Systems Cybersecurity Consultant', 'Information Security Consultant', 'Offensive Security Engineer', 'Physical Security Engineer', 'Security Analyst', 'Security Engineer', 'IT Security Analyst (Infor

In [48]:
# Convert Multiple into an explicit combined label
job_titles.loc[
    job_titles["NLP_Category"] == "Multiple",
    "NLP_Category"
] = "AI + Data Analytics"

# Keep only classified titles
nlp_classified_titles = job_titles[
    job_titles["NLP_Category"].notna()
].copy()

# Save
nlp_output = (
    processed_path / "nlp_job_title_classification.csv"
)

nlp_classified_titles[
    [
        "O*NET-SOC Code",
        "Title",
        "Job Title",
        "Clean_Job_Title",
        "NLP_Category"
    ]
].to_csv(
    nlp_output,
    index=False
)

print("Saved:", nlp_output)
print("Shape:", nlp_classified_titles.shape)

print("\nFinal category counts:")
print(
    nlp_classified_titles["NLP_Category"]
    .value_counts()
)

Saved: D:\MScProjects\labour-market-intelligence\data\processed\nlp_job_title_classification.csv
Shape: (196, 7)

Final category counts:
NLP_Category
Cybersecurity          116
Data Analytics          67
AI                      11
AI + Data Analytics      2
Name: count, dtype: int64


In [44]:
tk_demand = pd.read_excel(
    textkernel_path,
    sheet_name="3.Occupation 4-Digit NSA",
    header=4
)

print("Shape:", tk_demand.shape)

print("\nFirst 10 columns:")
print(tk_demand.columns[:10].tolist())

print("\nDate range:")
print(tk_demand["Month"].min())
print(tk_demand["Month"].max())

tk_demand.head()

Shape: (102, 415)

First 10 columns:
['Month', 'All Occupations', '1111 Chief executives and senior officials', '1112 Elected officers and representatives', '1121 Production managers and directors in manufacturing', '1122 Production managers and directors in construction', '1123 Production managers and directors in mining and energy', '1131 Financial managers and directors', '1132 Marketing, sales and advertising directors', '1133 Public relations and communications directors']

Date range:
2018-01-01 00:00:00
2026-06-01 00:00:00


,Month,All Occupations,1111 Chief executives and senior officials,1112 Elected officers and representatives,1121 Production managers and directors in manufacturing,1122 Production managers and directors in construction,1123 Production managers and directors in mining and energy,1131 Financial managers and directors,"1132 Marketing, sales and advertising directors",1133 Public relations and communications directors,...,9259 Elementary storage occupations n.e.c.,9261 Bar and catering supervisors,9262 Hospital porters,9263 Kitchen and catering assistants,9264 Waiters and waitresses,9265 Bar staff,9266 Coffee shop workers,9267 Leisure and theme park attendants,9269 Other elementary services occupations n.e.c.,Unknown
0,2018-01-01,993156,944,643,5355,3893,151,9622,3887,475,...,703,1559,79,9633,3096,2867,635,1389,828,1248
1,2018-02-01,854140,811,504,4420,3080,259,8053,3242,386,...,654,1293,59,11202,2847,2775,420,1516,880,1090
2,2018-03-01,921648,732,632,5121,3715,173,8532,3807,435,...,701,1571,65,12407,3253,3246,497,1463,930,1041
3,2018-04-01,819751,651,630,4355,3014,242,7502,2993,388,...,618,1333,48,8773,2851,2885,463,2139,748,1007
4,2018-05-01,891434,679,1330,4570,3552,151,8155,3327,393,...,611,1495,52,9488,2934,2749,417,1768,724,1168


In [49]:
# Remove columns that are not individual SOC occupations
occupation_cols = [
    col for col in tk_demand.columns
    if col not in ["Month", "All Occupations", "Unknown"]
]

# Wide -> long format
tk_long = tk_demand.melt(
    id_vars=["Month"],
    value_vars=occupation_cols,
    var_name="SOC_Occupation",
    value_name="Job_Adverts"
)

# Extract 4-digit SOC code and occupation name
tk_long["SOC_Code"] = (
    tk_long["SOC_Occupation"]
    .astype(str)
    .str.extract(r"^(\d{4})")[0]
)

tk_long["Occupation"] = (
    tk_long["SOC_Occupation"]
    .astype(str)
    .str.replace(r"^\d{4}\s+", "", regex=True)
)

# Ensure correct data types
tk_long["Month"] = pd.to_datetime(tk_long["Month"])

tk_long["Job_Adverts"] = pd.to_numeric(
    tk_long["Job_Adverts"],
    errors="coerce"
)

# Keep useful columns
tk_long = tk_long[
    ["Month", "SOC_Code", "Occupation", "Job_Adverts"]
].copy()

print("Shape:", tk_long.shape)

print("\nMissing SOC codes:")
print(tk_long["SOC_Code"].isna().sum())

print("\nUnique SOC occupations:")
print(tk_long["SOC_Code"].nunique())

print("\nDate range:")
print(tk_long["Month"].min(), "to", tk_long["Month"].max())

tk_long.head(10)

Shape: (42024, 4)

Missing SOC codes:
0

Unique SOC occupations:
412

Date range:
2018-01-01 00:00:00 to 2026-06-01 00:00:00


,Month,SOC_Code,Occupation,Job_Adverts
0,2018-01-01,1111,Chief executives and senior officials,944.0
1,2018-02-01,1111,Chief executives and senior officials,811.0
2,2018-03-01,1111,Chief executives and senior officials,732.0
3,2018-04-01,1111,Chief executives and senior officials,651.0
4,2018-05-01,1111,Chief executives and senior officials,679.0
5,2018-06-01,1111,Chief executives and senior officials,820.0
6,2018-07-01,1111,Chief executives and senior officials,711.0
7,2018-08-01,1111,Chief executives and senior officials,737.0
8,2018-09-01,1111,Chief executives and senior officials,791.0
9,2018-10-01,1111,Chief executives and senior officials,786.0


In [50]:
digital_terms = (
    r"software|programmer|developer|"
    r"data|database|"
    r"cyber|security|"
    r"information technology|IT |"
    r"computer|network|"
    r"web|digital"
)

candidate_occupations = (
    tk_long[
        tk_long["Occupation"]
        .str.contains(
            digital_terms,
            case=False,
            na=False,
            regex=True
        )
    ][["SOC_Code", "Occupation"]]
    .drop_duplicates()
    .sort_values("SOC_Code")
)

print("Candidate occupations:", len(candidate_occupations))

candidate_occupations.to_string(index=False)

Candidate occupations: 22


'SOC_Code                                             Occupation\n    1137                       Information technology directors\n    2131                                    IT project managers\n    2132                                            IT managers\n    2133 IT business analysts, architects and systems designers\n    2134     Programmers and software development professionals\n    2135                           Cyber security professionals\n    2136                   IT quality and testing professionals\n    2137                               IT network professionals\n    2139            Information technology professionals n.e.c.\n    2141                               Web design professionals\n    3131                              IT operations technicians\n    3132                            IT user support technicians\n    3133    Database administrators and web content technicians\n    3544                                          Data analysts\n    3573                

In [51]:
uk_soc_skill_map = {
    # Cybersecurity
    "2135": "Cybersecurity",

    # Data Analytics
    "3544": "Data Analytics",

    # Broader digital occupations relevant to AI/software
    # Keep separate rather than falsely calling all of them AI
    "2134": "Software Development"
}

tk_digital = tk_long[
    tk_long["SOC_Code"].isin(uk_soc_skill_map.keys())
].copy()

tk_digital["Skill_Category"] = (
    tk_digital["SOC_Code"]
    .map(uk_soc_skill_map)
)

print(
    tk_digital[
        ["SOC_Code", "Occupation", "Skill_Category"]
    ]
    .drop_duplicates()
)

print("\nRows:", len(tk_digital))

print("\nDate range:")
print(
    tk_digital["Month"].min(),
    "to",
    tk_digital["Month"].max()
)

      SOC_Code                                         Occupation  \
6018      2134  Programmers and software development professio...   
6120      2135                       Cyber security professionals   
19482     3544                                      Data analysts   

             Skill_Category  
6018   Software Development  
6120          Cybersecurity  
19482        Data Analytics  

Rows: 306

Date range:
2018-01-01 00:00:00 to 2026-06-01 00:00:00


In [52]:
# Keep the analysis period aligned to 2023–2025
demand_2325 = tk_digital[
    tk_digital["Month"].dt.year.isin([2023, 2024, 2025])
].copy()

demand_2325["Year"] = demand_2325["Month"].dt.year

annual_demand = (
    demand_2325
    .groupby(
        ["Year", "SOC_Code", "Occupation", "Skill_Category"],
        as_index=False
    )
    .agg(
        Annual_Job_Adverts=("Job_Adverts", "sum"),
        Average_Monthly_Adverts=("Job_Adverts", "mean")
    )
)

annual_demand["Average_Monthly_Adverts"] = (
    annual_demand["Average_Monthly_Adverts"].round(1)
)

annual_demand

,Year,SOC_Code,Occupation,Skill_Category,Annual_Job_Adverts,Average_Monthly_Adverts
0,2023,2134,Programmers and software development professio...,Software Development,194205.0,16183.8
1,2023,2135,Cyber security professionals,Cybersecurity,29493.0,2457.8
2,2023,3544,Data analysts,Data Analytics,29599.0,2466.6
3,2024,2134,Programmers and software development professio...,Software Development,98969.0,8247.4
4,2024,2135,Cyber security professionals,Cybersecurity,17143.0,1428.6
5,2024,3544,Data analysts,Data Analytics,17516.0,1459.7
6,2025,2134,Programmers and software development professio...,Software Development,113816.0,9484.7
7,2025,2135,Cyber security professionals,Cybersecurity,18460.0,1846.0
8,2025,3544,Data analysts,Data Analytics,19877.0,1656.4


In [53]:
coverage = (
    demand_2325
    .groupby(["Year", "SOC_Code", "Skill_Category"])
    .agg(
        Months_Total=("Month", "count"),
        Months_With_Data=("Job_Adverts", "count"),
        Missing_Values=("Job_Adverts", lambda x: x.isna().sum())
    )
    .reset_index()
)

coverage

,Year,SOC_Code,Skill_Category,Months_Total,Months_With_Data,Missing_Values
0,2023,2134,Software Development,12,12,0
1,2023,2135,Cybersecurity,12,12,0
2,2023,3544,Data Analytics,12,12,0
3,2024,2134,Software Development,12,12,0
4,2024,2135,Cybersecurity,12,12,0
5,2024,3544,Data Analytics,12,12,0
6,2025,2134,Software Development,12,12,0
7,2025,2135,Cybersecurity,12,10,2
8,2025,3544,Data Analytics,12,12,0


In [54]:
cyber_missing = demand_2325[
    (demand_2325["SOC_Code"] == "2135") &
    (demand_2325["Job_Adverts"].isna())
][
    ["Month", "SOC_Code", "Occupation", "Job_Adverts"]
]

cyber_missing

,Month,SOC_Code,Occupation,Job_Adverts
6209,2025-06-01,2135,Cyber security professionals,NaN
6210,2025-07-01,2135,Cyber security professionals,NaN


In [55]:
# Calculate annual statistics using observed months
demand_summary = (
    demand_2325
    .groupby(
        ["Year", "SOC_Code", "Occupation", "Skill_Category"],
        as_index=False
    )
    .agg(
        Observed_Months=("Job_Adverts", "count"),
        Total_Job_Adverts=("Job_Adverts", "sum"),
        Average_Monthly_Adverts=("Job_Adverts", "mean")
    )
)

demand_summary["Average_Monthly_Adverts"] = (
    demand_summary["Average_Monthly_Adverts"].round(1)
)

# YoY based on average monthly demand
demand_summary = demand_summary.sort_values(
    ["SOC_Code", "Year"]
)

demand_summary["YoY_Demand_Growth_Pct"] = (
    demand_summary
    .groupby("SOC_Code")["Average_Monthly_Adverts"]
    .pct_change() * 100
).round(2)

demand_summary

,Year,SOC_Code,Occupation,Skill_Category,Observed_Months,Total_Job_Adverts,Average_Monthly_Adverts,YoY_Demand_Growth_Pct
0,2023,2134,Programmers and software development professio...,Software Development,12,194205.0,16183.8,NaN
3,2024,2134,Programmers and software development professio...,Software Development,12,98969.0,8247.4,-49.04
6,2025,2134,Programmers and software development professio...,Software Development,12,113816.0,9484.7,15.00
1,2023,2135,Cyber security professionals,Cybersecurity,12,29493.0,2457.8,NaN
4,2024,2135,Cyber security professionals,Cybersecurity,12,17143.0,1428.6,-41.87
7,2025,2135,Cyber security professionals,Cybersecurity,10,18460.0,1846.0,29.22
2,2023,3544,Data analysts,Data Analytics,12,29599.0,2466.6,NaN
5,2024,3544,Data analysts,Data Analytics,12,17516.0,1459.7,-40.82
8,2025,3544,Data analysts,Data Analytics,12,19877.0,1656.4,13.48


In [56]:
output_path = (
    processed_path / "uk_digital_job_demand_2023_2025.csv"
)

demand_summary.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Shape:", demand_summary.shape)

Saved: D:\MScProjects\labour-market-intelligence\data\processed\uk_digital_job_demand_2023_2025.csv
Shape: (9, 8)


In [57]:
# Search all Excel/CSV filenames for anything AI-related
for p in raw_path.rglob("*"):
    if p.suffix.lower() in [".csv", ".xlsx", ".xls"]:
        name = p.name.lower()

        if any(term in name for term in [
            "artificial",
            "intelligence",
            "machine",
            "ai",
            "skill"
        ]):
            print(p)

D:\MScProjects\labour-market-intelligence\data\raw\onet\Essential Skills.xlsx
D:\MScProjects\labour-market-intelligence\data\raw\onet\Software Skills.xlsx
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Occupation_2023\Occupation SOC20 (4) Table 14.10a   Paid hours worked - Basic 2023.xlsx
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Occupation_2023\Occupation SOC20 (4) Table 14.10b   Paid hours worked - Basic 2023 CV.xlsx
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Occupation_2023\Occupation SOC20 (4) Table 14.11a   Paid hours worked - Overtime 2023.xlsx
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Occupation_2023\Occupation SOC20 (4) Table 14.11b   Paid hours worked - Overtime 2023 CV.xlsx
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Occupation_2023\Occupation SOC20 (4) Table 14.9a   Paid hours worked - Total 2023.xlsx
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Occupation

In [58]:
ashe_occ_path = raw_path / "ons"

for year in [2023, 2024, 2025]:
    print(f"\n--- {year} ---")

    folder = ashe_occ_path / f"ONS_ASHE_Occupation_{year}"

    for p in folder.rglob("*.xlsx"):
        name = p.name.lower()

        if "14.7a" in name or "annual pay - gross" in name:
            print(p)


--- 2023 ---
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Occupation_2023\Occupation SOC20 (4) Table 14.7a   Annual pay - Gross 2023.xlsx
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Occupation_2023\Occupation SOC20 (4) Table 14.7b   Annual pay - Gross 2023 CV.xlsx

--- 2024 ---
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Occupation_2024\Occupation SOC20 (4) Table 14.7a   Annual pay - Gross 2024.xlsx
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Occupation_2024\Occupation SOC20 (4) Table 14.7b   Annual pay - Gross 2024 CV.xlsx

--- 2025 ---
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Occupation_2025\PROV - Occupation SOC20 (4) Table 14.7a   Annual pay - Gross 2025.xlsx
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Occupation_2025\PROV - Occupation SOC20 (4) Table 14.7b   Annual pay - Gross 2025 CV.xlsx


In [59]:
salary_2023_path = (
    raw_path / "ons" / "ONS_ASHE_Occupation_2023" /
    "Occupation SOC20 (4) Table 14.7a   Annual pay - Gross 2023.xlsx"
)

salary_xls = pd.ExcelFile(salary_2023_path)

print(salary_xls.sheet_names)

['Notes', 'All', 'Male', 'Female', 'Full-Time', 'Part-Time', 'Male Full-Time', 'Male Part-Time', 'Female Full-Time', 'Female Part-Time']


In [60]:
salary_2023_raw = pd.read_excel(
    salary_2023_path,
    sheet_name="All",
    header=None
)

print("Shape:", salary_2023_raw.shape)

salary_2023_raw.head(25)

Shape: (563, 20)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,Table 14.7a Annual pay - Gross (£) - For all...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,Number,NaN,Annual,NaN,Annual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,of jobsb,NaN,percentage,NaN,percentage,Percentiles,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Description,Code,(thousand),Median,change,Mean,change,10,20,25,30,40,60,70,75,80,90,NaN,NaN,NaN
5,All employees,NaN,22783,29511,6.3,35393,5.8,10050,16461,19437,21693,25598,34244,40000,43160,47283,60776,NaN,NaN,NaN
6,"Managers, directors and senior officials",1,2563,45702,8.3,61946,7,19691,27118,30000,32900,39052,53283,63804,71357,80000,108055,NaN,Key,Statistical robustness
7,Corporate managers and directors,11,2033,50498,9.8,67815,8.1,20061,28537,31893,35721,42858,59684,71733,78554,88870,117349,NaN,CV <= 5%,Estimates are considered precise
8,Chief Executives and Senior Officials,111,102,76653,14.4,113281,8.2,x,x,x,x,63387,90560,x,x,x,x,NaN,CV > 5% and <= 10%,Estimates are considered reasonably precise
9,Chief executives and senior officials,1111,96,80259,10.9,118729,7.7,x,x,x,x,66025,92832,x,x,x,x,NaN,CV > 10% and <= 20%,Estimates are considered acceptable


In [61]:
salary_2023 = salary_2023_raw.iloc[5:, [0, 1, 3]].copy()

salary_2023.columns = [
    "Occupation",
    "SOC_Code",
    "Median_Annual_Salary"
]

salary_2023["SOC_Code"] = (
    salary_2023["SOC_Code"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
)

target_salary_2023 = salary_2023[
    salary_2023["SOC_Code"].isin(
        ["2134", "2135", "3544"]
    )
].copy()

target_salary_2023

,Occupation,SOC_Code,Median_Annual_Salary
85,Programmers and software development profes...,2134,49444
86,Cyber security professionals,2135,45573
260,Data analysts,3544,33695


In [62]:
salary_rows = []

for year in [2023, 2024, 2025]:

    folder = raw_path / "ons" / f"ONS_ASHE_Occupation_{year}"

    # Find 14.7a file only — exclude CV/7b
    files = [
        p for p in folder.rglob("*.xlsx")
        if "14.7a" in p.name.lower()
    ]

    file_path = files[0]

    raw = pd.read_excel(
        file_path,
        sheet_name="All",
        header=None
    )

    temp = raw.iloc[5:, [0, 1, 3]].copy()

    temp.columns = [
        "Occupation",
        "SOC_Code",
        "Median_Annual_Salary"
    ]

    temp["SOC_Code"] = (
        temp["SOC_Code"]
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )

    temp = temp[
        temp["SOC_Code"].isin(
            ["2134", "2135", "3544"]
        )
    ].copy()

    temp["Year"] = year

    salary_rows.append(temp)


digital_salary = pd.concat(
    salary_rows,
    ignore_index=True
)

digital_salary = digital_salary[
    [
        "Year",
        "SOC_Code",
        "Occupation",
        "Median_Annual_Salary"
    ]
]

digital_salary

,Year,SOC_Code,Occupation,Median_Annual_Salary
0,2023,2134,Programmers and software development profes...,49444
1,2023,2135,Cyber security professionals,45573
2,2023,3544,Data analysts,33695
3,2024,2134,Programmers and software development profes...,53363
4,2024,2135,Cyber security professionals,48555
5,2024,3544,Data analysts,34125
6,2025,2134,Programmers and software development profes...,55587
7,2025,2135,Cyber security professionals,54816
8,2025,3544,Data analysts,38107


In [63]:
digital_salary["Median_Annual_Salary"] = pd.to_numeric(
    digital_salary["Median_Annual_Salary"],
    errors="coerce"
)

digital_salary = digital_salary.sort_values(
    ["SOC_Code", "Year"]
)

# Salary YoY growth
digital_salary["Salary_Growth_Pct"] = (
    digital_salary
    .groupby("SOC_Code")["Median_Annual_Salary"]
    .pct_change() * 100
).round(2)

# Join to our validated demand results
integrated_digital = demand_summary.merge(
    digital_salary[
        [
            "Year",
            "SOC_Code",
            "Median_Annual_Salary",
            "Salary_Growth_Pct"
        ]
    ],
    on=["Year", "SOC_Code"],
    how="left"
)

integrated_digital[
    [
        "Year",
        "SOC_Code",
        "Skill_Category",
        "Average_Monthly_Adverts",
        "YoY_Demand_Growth_Pct",
        "Median_Annual_Salary",
        "Salary_Growth_Pct",
        "Observed_Months"
    ]
]

,Year,SOC_Code,Skill_Category,Average_Monthly_Adverts,YoY_Demand_Growth_Pct,Median_Annual_Salary,Salary_Growth_Pct,Observed_Months
0,2023,2134,Software Development,16183.8,NaN,49444,NaN,12
1,2024,2134,Software Development,8247.4,-49.04,53363,7.93,12
2,2025,2134,Software Development,9484.7,15.00,55587,4.17,12
3,2023,2135,Cybersecurity,2457.8,NaN,45573,NaN,12
4,2024,2135,Cybersecurity,1428.6,-41.87,48555,6.54,12
5,2025,2135,Cybersecurity,1846.0,29.22,54816,12.89,10
6,2023,3544,Data Analytics,2466.6,NaN,33695,NaN,12
7,2024,3544,Data Analytics,1459.7,-40.82,34125,1.28,12
8,2025,3544,Data Analytics,1656.4,13.48,38107,11.67,12


In [64]:
integrated_output = (
    processed_path / "integrated_digital_demand_salary_2023_2025.csv"
)

integrated_digital.to_csv(
    integrated_output,
    index=False
)

print("Saved:", integrated_output)
print("Shape:", integrated_digital.shape)

Saved: D:\MScProjects\labour-market-intelligence\data\processed\integrated_digital_demand_salary_2023_2025.csv
Shape: (9, 10)


In [65]:
summary_2325 = (
    integrated_digital[
        integrated_digital["Year"].isin([2023, 2025])
    ]
    .pivot(
        index=["SOC_Code", "Skill_Category"],
        columns="Year",
        values=[
            "Average_Monthly_Adverts",
            "Median_Annual_Salary"
        ]
    )
)

summary_2325.columns = [
    f"{metric}_{year}"
    for metric, year in summary_2325.columns
]

summary_2325 = summary_2325.reset_index()

summary_2325["Demand_Change_2023_2025_Pct"] = (
    (
        summary_2325["Average_Monthly_Adverts_2025"]
        / summary_2325["Average_Monthly_Adverts_2023"]
        - 1
    ) * 100
).round(2)

summary_2325["Salary_Change_2023_2025_Pct"] = (
    (
        summary_2325["Median_Annual_Salary_2025"]
        / summary_2325["Median_Annual_Salary_2023"]
        - 1
    ) * 100
).round(2)

summary_2325

,SOC_Code,Skill_Category,Average_Monthly_Adverts_2023,Average_Monthly_Adverts_2025,Median_Annual_Salary_2023,Median_Annual_Salary_2025,Demand_Change_2023_2025_Pct,Salary_Change_2023_2025_Pct
0,2134,Software Development,16183.8,9484.7,49444.0,55587.0,-41.39,12.42
1,2135,Cybersecurity,2457.8,1846.0,45573.0,54816.0,-24.89,20.28
2,3544,Data Analytics,2466.6,1656.4,33695.0,38107.0,-32.85,13.09


In [66]:
summary_output = (
    processed_path / "digital_demand_salary_summary_2023_2025.csv"
)

summary_2325.to_csv(
    summary_output,
    index=False
)

print("Saved:", summary_output)
print("Shape:", summary_2325.shape)

Saved: D:\MScProjects\labour-market-intelligence\data\processed\digital_demand_salary_summary_2023_2025.csv
Shape: (3, 8)


In [67]:
powerbi_data = integrated_digital[
    [
        "Year",
        "SOC_Code",
        "Occupation",
        "Skill_Category",
        "Average_Monthly_Adverts",
        "YoY_Demand_Growth_Pct",
        "Median_Annual_Salary",
        "Salary_Growth_Pct",
        "Observed_Months"
    ]
].copy()

# Flag incomplete observations
powerbi_data["Data_Quality"] = powerbi_data.apply(
    lambda row: (
        "Incomplete - 10 months"
        if row["SOC_Code"] == "2135"
        and row["Year"] == 2025
        else "Complete"
    ),
    axis=1
)

powerbi_output = (
    processed_path / "powerbi_digital_labour_market.csv"
)

powerbi_data.to_csv(
    powerbi_output,
    index=False
)

print("Saved:", powerbi_output)
print("Shape:", powerbi_data.shape)

powerbi_data

Saved: D:\MScProjects\labour-market-intelligence\data\processed\powerbi_digital_labour_market.csv
Shape: (9, 10)


,Year,SOC_Code,Occupation,Skill_Category,Average_Monthly_Adverts,YoY_Demand_Growth_Pct,Median_Annual_Salary,Salary_Growth_Pct,Observed_Months,Data_Quality
0,2023,2134,Programmers and software development professio...,Software Development,16183.8,NaN,49444,NaN,12,Complete
1,2024,2134,Programmers and software development professio...,Software Development,8247.4,-49.04,53363,7.93,12,Complete
2,2025,2134,Programmers and software development professio...,Software Development,9484.7,15.00,55587,4.17,12,Complete
3,2023,2135,Cyber security professionals,Cybersecurity,2457.8,NaN,45573,NaN,12,Complete
4,2024,2135,Cyber security professionals,Cybersecurity,1428.6,-41.87,48555,6.54,12,Complete
5,2025,2135,Cyber security professionals,Cybersecurity,1846.0,29.22,54816,12.89,10,Incomplete - 10 months
6,2023,3544,Data analysts,Data Analytics,2466.6,NaN,33695,NaN,12,Complete
7,2024,3544,Data analysts,Data Analytics,1459.7,-40.82,34125,1.28,12,Complete
8,2025,3544,Data analysts,Data Analytics,1656.4,13.48,38107,11.67,12,Complete


In [68]:
from pathlib import Path

base = Path(
    r"D:\MScProjects\labour-market-intelligence\data\raw\ons"
    r"\ONS_ASHE_Region_Occupation_2025"
)

files = list(base.rglob("*.xlsx"))

annual_pay_files = [
    f for f in files
    if "7a" in f.name.lower()
    and "annual pay" in f.name.lower()
]

print("Annual pay files found:", len(annual_pay_files))

for f in annual_pay_files:
    print(f)

Annual pay files found: 2
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Region_Occupation_2025\ashetable152025provisional\ASHE Table 15 (3) 2025 Provisional\PROV - Work Region Occupation SOC20 (3) Table 15 (3).7a   Annual pay - Gross 2025.xlsx
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Region_Occupation_2025\ashetable152025provisional\ASHE Table 15 (4) 2025 Provisional\PROV - Work Region Occupation SOC20 (4) Table 15 (4).7a   Annual pay - Gross 2025.xlsx


In [69]:
import pandas as pd

region_salary_2025_path = (
    base
    / "ashetable152025provisional"
    / "ASHE Table 15 (4) 2025 Provisional"
    / "PROV - Work Region Occupation SOC20 (4) Table 15 (4).7a   Annual pay - Gross 2025.xlsx"
)

region_xls = pd.ExcelFile(region_salary_2025_path)

print(region_xls.sheet_names)
region_salary_2025_path = (
    base
    / "ashetable152025provisional"
    / "ASHE Table 15 (4) 2025 Provisional"
    / "PROV - Work Region Occupation SOC20 (4) Table 15 (4).7a   Annual pay - Gross 2025.xlsx"
)

region_xls = pd.ExcelFile(region_salary_2025_path)

print(region_xls.sheet_names)

['Notes', 'All', 'Male', 'Female', 'Full-Time', 'Part-Time', 'Male Full-Time', 'Male Part-Time', 'Female Full-Time', 'Female Part-Time']
['Notes', 'All', 'Male', 'Female', 'Full-Time', 'Part-Time', 'Male Full-Time', 'Male Part-Time', 'Female Full-Time', 'Female Part-Time']


In [70]:
region_2025_raw = pd.read_excel(
    region_salary_2025_path,
    sheet_name="All",
    header=None
)

print("Shape:", region_2025_raw.shape)

region_2025_raw.head(20)

Shape: (4542, 20)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,Table 15 (4).7a Annual pay - Gross (£) - For...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,Number,NaN,Annual,NaN,Annual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,of jobsb,NaN,percentage,NaN,percentage,Percentiles,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Description,Code,(thousand),Median,change,Mean,change,10,20,25,30,40,60,70,75,80,90,NaN,NaN,NaN
5,"North East, Chief executives and senior ...",1111,x,x,NaN,x,NaN,x,x,x,x,x,x,x,x,x,x,NaN,NaN,NaN
6,"North East, Elected officers and represe...",1112,:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Key,Statistical robustness
7,"North East, Production managers and dire...",1121,13,x,NaN,59815,8.4,x,x,x,x,x,x,x,x,x,x,NaN,CV <= 5%,Estimates are considered precise
8,"North East, Production managers and dire...",1122,x,x,NaN,x,NaN,x,x,x,x,x,x,x,x,x,x,NaN,CV > 5% and <= 10%,Estimates are considered reasonably precise
9,"North East, Production managers and dire...",1123,..,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CV > 10% and <= 20%,Estimates are considered acceptable


In [71]:
region_2025 = region_2025_raw.iloc[5:, [0, 1, 3]].copy()

region_2025.columns = [
    "Region_Occupation",
    "SOC_Code",
    "Median_Annual_Salary"
]

region_2025["SOC_Code"] = (
    region_2025["SOC_Code"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
)

target_region_2025 = region_2025[
    region_2025["SOC_Code"].isin(
        ["2134", "2135", "3544"]
    )
].copy()

print("Rows found:", len(target_region_2025))

print(
    target_region_2025.to_string(index=False)
)

Rows found: 33
                                                                  Region_Occupation SOC_Code Median_Annual_Salary
                    North East, Programmers and software development professionals      2134                54279
                                           North East, Cyber security professionals     2135                  NaN
                                                          North East, Data analysts     3544                38225
                    North West, Programmers and software development professionals      2134                49505
                                           North West, Cyber security professionals     2135                51954
                                                          North West, Data analysts     3544                33871
      Yorkshire and The Humber, Programmers and software development professionals      2134                46594
                             Yorkshire and The Humber, Cyber security pro

In [72]:
import numpy as np

regional_focus_2025 = target_region_2025.copy()

# Extract region from "Region, Occupation"
regional_focus_2025["Region"] = (
    regional_focus_2025["Region_Occupation"]
    .str.split(",", n=1)
    .str[0]
    .str.strip()
)

# Convert salary to numeric.
# ONS suppression/unreliable markers such as x become NaN, NOT zero.
regional_focus_2025["Median_Annual_Salary"] = pd.to_numeric(
    regional_focus_2025["Median_Annual_Salary"],
    errors="coerce"
)

# Map SOC codes to our project categories
category_map = {
    "2134": "Software Development",
    "2135": "Cybersecurity",
    "3544": "Data Analytics"
}

regional_focus_2025["Skill_Category"] = (
    regional_focus_2025["SOC_Code"].map(category_map)
)

# Keep proposal-relevant geographical contexts
regional_focus_2025 = regional_focus_2025[
    regional_focus_2025["Region"].isin(
        ["London", "North West", "West Midlands"]
    )
].copy()

# Add interpretation label
location_map = {
    "London": "London",
    "North West": "Manchester regional context",
    "West Midlands": "Birmingham regional context"
}

regional_focus_2025["Location_Context"] = (
    regional_focus_2025["Region"].map(location_map)
)

regional_focus_2025["Year"] = 2025

regional_focus_2025 = regional_focus_2025[
    [
        "Year",
        "Region",
        "Location_Context",
        "SOC_Code",
        "Skill_Category",
        "Median_Annual_Salary"
    ]
].sort_values(
    ["Skill_Category", "Region"]
)

regional_focus_2025

,Year,Region,Location_Context,SOC_Code,Skill_Category,Median_Annual_Salary
2949,2025,London,London,2135,Cybersecurity,59590.0
477,2025,North West,Manchester regional context,2135,Cybersecurity,51954.0
1713,2025,West Midlands,Birmingham regional context,2135,Cybersecurity,NaN
3080,2025,London,London,3544,Data Analytics,42850.0
608,2025,North West,Manchester regional context,3544,Data Analytics,33871.0
1844,2025,West Midlands,Birmingham regional context,3544,Data Analytics,44172.0
2948,2025,London,London,2134,Software Development,74284.0
476,2025,North West,Manchester regional context,2134,Software Development,49505.0
1712,2025,West Midlands,Birmingham regional context,2134,Software Development,47744.0


In [73]:
from pathlib import Path
import pandas as pd

ons_base = Path(
    r"D:\MScProjects\labour-market-intelligence\data\raw\ons"
)

# Find the SOC20 (4) regional annual gross-pay 7a file for each year
region_files = {}

for year in [2023, 2024, 2025]:
    folder = ons_base / f"ONS_ASHE_Region_Occupation_{year}"

    matches = [
        f for f in folder.rglob("*.xlsx")
        if "15 (4).7a" in f.name.lower()
        and "annual pay" in f.name.lower()
        and "cv" not in f.name.lower()
    ]

    print(f"\n--- {year} ---")
    for f in matches:
        print(f)

    if len(matches) == 1:
        region_files[year] = matches[0]
    else:
        print(f"WARNING: expected 1 file, found {len(matches)}")
category_map = {
    "2134": "Software Development",
    "2135": "Cybersecurity",
    "3544": "Data Analytics"
}

location_map = {
    "London": "London",
    "North West": "Manchester regional context",
    "West Midlands": "Birmingham regional context"
}

def extract_regional_salary(file_path, year):

    raw = pd.read_excel(
        file_path,
        sheet_name="All",
        header=None
    )

    # ONS data begins after header rows
    df = raw.iloc[5:, [0, 1, 3]].copy()

    df.columns = [
        "Region_Occupation",
        "SOC_Code",
        "Median_Annual_Salary"
    ]

    df["SOC_Code"] = (
        df["SOC_Code"]
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )

    # Keep target digital occupations
    df = df[
        df["SOC_Code"].isin(["2134", "2135", "3544"])
    ].copy()

    # Extract region
    df["Region"] = (
        df["Region_Occupation"]
        .astype(str)
        .str.split(",", n=1)
        .str[0]
        .str.strip()
    )

    # Keep proposal-relevant regional contexts
    df = df[
        df["Region"].isin(
            ["London", "North West", "West Midlands"]
        )
    ].copy()

    # Suppressed/unreliable ONS values become missing
    df["Median_Annual_Salary"] = pd.to_numeric(
        df["Median_Annual_Salary"],
        errors="coerce"
    )

    df["Skill_Category"] = df["SOC_Code"].map(category_map)
    df["Location_Context"] = df["Region"].map(location_map)
    df["Year"] = year

    return df[
        [
            "Year",
            "Region",
            "Location_Context",
            "SOC_Code",
            "Skill_Category",
            "Median_Annual_Salary"
        ]
    ]


--- 2023 ---
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Region_Occupation_2023\ASHE Table 15 (4) 2023 Revised\Work Region Occupation SOC20 (4) Table 15 (4).7a   Annual pay - Gross 2023.xlsx

--- 2024 ---
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Region_Occupation_2024\Table 15 (4)\Work Region Occupation SOC20 (4) Table 15 (4).7a   Annual pay - Gross 2024.xlsx

--- 2025 ---
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Region_Occupation_2025\ashetable152025provisional\ASHE Table 15 (4) 2025 Provisional\PROV - Work Region Occupation SOC20 (4) Table 15 (4).7a   Annual pay - Gross 2025.xlsx


In [10]:
from pathlib import Path

folder_2024 = Path(
    r"D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Region_Occupation_2024"
)

for f in folder_2024.rglob("*.xlsx"):
    if "annual pay" in f.name.lower():
        print(f)

In [11]:
folder_2024 = Path(
    r"D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Region_Occupation_2024"
)

print("Folder exists:", folder_2024.exists())
print("\nAll files inside 2024 folder:")

for f in folder_2024.rglob("*"):
    if f.is_file():
        print(f)

Folder exists: True

All files inside 2024 folder:
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Region_Occupation_2024\Table 15 (3).zip
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Region_Occupation_2024\Table 15 (4).zip


In [74]:
import zipfile
from pathlib import Path

zip_2024 = Path(
    r"D:\MScProjects\labour-market-intelligence\data\raw\ons"
    r"\ONS_ASHE_Region_Occupation_2024\Table 15 (4).zip"
)

with zipfile.ZipFile(zip_2024, "r") as z:
    names = z.namelist()

    print("Files in ZIP:", len(names))
    print()

    for name in names:
        if "7a" in name.lower() or "annual pay" in name.lower():
            print(name)

Files in ZIP: 23

Table 15 (4)/Work Region Occupation SOC20 (4) Table 15 (4).7a   Annual pay - Gross 2024.xlsx
Table 15 (4)/Work Region Occupation SOC20 (4) Table 15 (4).7b   Annual pay - Gross 2024 CV.xlsx
Table 15 (4)/Work Region Occupation SOC20 (4) Table 15 (4).8a   Annual pay - Incentive 2024.xlsx
Table 15 (4)/Work Region Occupation SOC20 (4) Table 15 (4).8b   Annual pay - Incentive 2024 CV.xlsx


In [75]:
import zipfile
from pathlib import Path

zip_2024 = Path(
    r"D:\MScProjects\labour-market-intelligence\data\raw\ons"
    r"\ONS_ASHE_Region_Occupation_2024\Table 15 (4).zip"
)

extract_folder = Path(
    r"D:\MScProjects\labour-market-intelligence\data\raw\ons"
    r"\ONS_ASHE_Region_Occupation_2024"
)

target_file = (
    "Table 15 (4)/"
    "Work Region Occupation SOC20 (4) Table 15 (4).7a   "
    "Annual pay - Gross 2024.xlsx"
)

with zipfile.ZipFile(zip_2024, "r") as z:
    z.extract(target_file, extract_folder)

extracted_file = extract_folder / target_file

print("Extracted:", extracted_file.exists())
print(extracted_file)

Extracted: True
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Region_Occupation_2024\Table 15 (4)\Work Region Occupation SOC20 (4) Table 15 (4).7a   Annual pay - Gross 2024.xlsx


In [76]:
region_files = {}

for year in [2023, 2024, 2025]:
    folder = ons_base / f"ONS_ASHE_Region_Occupation_{year}"

    matches = [
        f for f in folder.rglob("*.xlsx")
        if "15 (4).7a" in f.name.lower()
        and "annual pay" in f.name.lower()
        and "cv" not in f.name.lower()
    ]

    print(f"\n--- {year} ---")
    for f in matches:
        print(f)

    if len(matches) == 1:
        region_files[year] = matches[0]
    else:
        print(f"WARNING: expected 1 file, found {len(matches)}")


--- 2023 ---
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Region_Occupation_2023\ASHE Table 15 (4) 2023 Revised\Work Region Occupation SOC20 (4) Table 15 (4).7a   Annual pay - Gross 2023.xlsx

--- 2024 ---
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Region_Occupation_2024\Table 15 (4)\Work Region Occupation SOC20 (4) Table 15 (4).7a   Annual pay - Gross 2024.xlsx

--- 2025 ---
D:\MScProjects\labour-market-intelligence\data\raw\ons\ONS_ASHE_Region_Occupation_2025\ashetable152025provisional\ASHE Table 15 (4) 2025 Provisional\PROV - Work Region Occupation SOC20 (4) Table 15 (4).7a   Annual pay - Gross 2025.xlsx


In [77]:
regional_salary_all = pd.concat(
    [
        extract_regional_salary(region_files[year], year)
        for year in [2023, 2024, 2025]
    ],
    ignore_index=True
)

regional_salary_all = (
    regional_salary_all
    .sort_values(["Year", "Skill_Category", "Region"])
    .reset_index(drop=True)
)

print("Shape:", regional_salary_all.shape)
print("\nMissing salary values:")
print(regional_salary_all[
    regional_salary_all["Median_Annual_Salary"].isna()
].to_string(index=False))

print("\nFull dataset:")
print(regional_salary_all.to_string(index=False))

Shape: (27, 6)

Missing salary values:
 Year        Region            Location_Context SOC_Code Skill_Category  Median_Annual_Salary
 2023        London                      London     2135  Cybersecurity                   NaN
 2024    North West Manchester regional context     2135  Cybersecurity                   NaN
 2024        London                      London     3544 Data Analytics                   NaN
 2025 West Midlands Birmingham regional context     2135  Cybersecurity                   NaN

Full dataset:
 Year        Region            Location_Context SOC_Code       Skill_Category  Median_Annual_Salary
 2023        London                      London     2135        Cybersecurity                   NaN
 2023    North West Manchester regional context     2135        Cybersecurity               59001.0
 2023 West Midlands Birmingham regional context     2135        Cybersecurity               44505.0
 2023        London                      London     3544       Data Analytic

In [78]:
regional_analysis = regional_salary_all.copy()

# West Midlands salary baseline for each occupation/year
wm_baseline = (
    regional_analysis[
        regional_analysis["Region"] == "West Midlands"
    ][
        ["Year", "SOC_Code", "Median_Annual_Salary"]
    ]
    .rename(
        columns={
            "Median_Annual_Salary": "West_Midlands_Baseline"
        }
    )
)

regional_analysis = regional_analysis.merge(
    wm_baseline,
    on=["Year", "SOC_Code"],
    how="left"
)

# Absolute salary difference
regional_analysis["Salary_Premium_GBP"] = (
    regional_analysis["Median_Annual_Salary"]
    - regional_analysis["West_Midlands_Baseline"]
)

# Percentage premium
regional_analysis["Salary_Premium_Pct"] = (
    regional_analysis["Salary_Premium_GBP"]
    / regional_analysis["West_Midlands_Baseline"]
    * 100
).round(2)

regional_analysis = regional_analysis.sort_values(
    ["Year", "Skill_Category", "Region"]
).reset_index(drop=True)

print(
    regional_analysis.to_string(index=False)
)

 Year        Region            Location_Context SOC_Code       Skill_Category  Median_Annual_Salary  West_Midlands_Baseline  Salary_Premium_GBP  Salary_Premium_Pct
 2023        London                      London     2135        Cybersecurity                   NaN                 44505.0                 NaN                 NaN
 2023    North West Manchester regional context     2135        Cybersecurity               59001.0                 44505.0             14496.0               32.57
 2023 West Midlands Birmingham regional context     2135        Cybersecurity               44505.0                 44505.0                 0.0                0.00
 2023        London                      London     3544       Data Analytics               40198.0                 31094.0              9104.0               29.28
 2023    North West Manchester regional context     3544       Data Analytics               31496.0                 31094.0               402.0                1.29
 2023 West Midla

In [79]:
regional_premium_summary = regional_analysis[
    regional_analysis["Region"].isin(
        ["London", "North West"]
    )
][
    [
        "Year",
        "Skill_Category",
        "Location_Context",
        "Median_Annual_Salary",
        "West_Midlands_Baseline",
        "Salary_Premium_GBP",
        "Salary_Premium_Pct"
    ]
].copy()

print(
    regional_premium_summary.to_string(index=False)
)

 Year       Skill_Category            Location_Context  Median_Annual_Salary  West_Midlands_Baseline  Salary_Premium_GBP  Salary_Premium_Pct
 2023        Cybersecurity                      London                   NaN                 44505.0                 NaN                 NaN
 2023        Cybersecurity Manchester regional context               59001.0                 44505.0             14496.0               32.57
 2023       Data Analytics                      London               40198.0                 31094.0              9104.0               29.28
 2023       Data Analytics Manchester regional context               31496.0                 31094.0               402.0                1.29
 2023 Software Development                      London               67505.0                 41315.0             26190.0               63.39
 2023 Software Development Manchester regional context               43207.0                 41315.0              1892.0                4.58
 2024        

In [18]:
salary_change = (
    regional_salary_all[
        regional_salary_all["Year"].isin([2023, 2025])
    ]
    .pivot_table(
        index=[
            "Region",
            "Location_Context",
            "SOC_Code",
            "Skill_Category"
        ],
        columns="Year",
        values="Median_Annual_Salary",
        aggfunc="first"
    )
    .reset_index()
)

salary_change.columns.name = None

salary_change = salary_change.rename(
    columns={
        2023: "Median_Salary_2023",
        2025: "Median_Salary_2025"
    }
)

salary_change["Salary_Change_GBP_2023_2025"] = (
    salary_change["Median_Salary_2025"]
    - salary_change["Median_Salary_2023"]
)

salary_change["Salary_Change_Pct_2023_2025"] = (
    (
        salary_change["Median_Salary_2025"]
        - salary_change["Median_Salary_2023"]
    )
    / salary_change["Median_Salary_2023"]
    * 100
).round(2)

salary_change = salary_change.sort_values(
    ["Skill_Category", "Region"]
).reset_index(drop=True)

print(salary_change.to_string(index=False))

       Region            Location_Context SOC_Code       Skill_Category  Median_Salary_2023  Median_Salary_2025  Salary_Change_GBP_2023_2025  Salary_Change_Pct_2023_2025
       London                      London     2135        Cybersecurity                 NaN             59590.0                          NaN                          NaN
   North West Manchester regional context     2135        Cybersecurity             59001.0             51954.0                      -7047.0                       -11.94
West Midlands Birmingham regional context     2135        Cybersecurity             44505.0                 NaN                          NaN                          NaN
       London                      London     3544       Data Analytics             40198.0             42850.0                       2652.0                         6.60
   North West Manchester regional context     3544       Data Analytics             31496.0             33871.0                       2375.0          

In [19]:
processed_path = Path(
    r"D:\MScProjects\labour-market-intelligence\data\processed"
)

regional_salary_all.to_csv(
    processed_path / "regional_digital_salary_2023_2025.csv",
    index=False
)

regional_analysis.to_csv(
    processed_path / "regional_salary_premiums_2023_2025.csv",
    index=False
)

salary_change.to_csv(
    processed_path / "regional_salary_change_2023_2025.csv",
    index=False
)

print("Saved regional datasets:")
print("1. regional_digital_salary_2023_2025.csv")
print("2. regional_salary_premiums_2023_2025.csv")
print("3. regional_salary_change_2023_2025.csv")

Saved regional datasets:
1. regional_digital_salary_2023_2025.csv
2. regional_salary_premiums_2023_2025.csv
3. regional_salary_change_2023_2025.csv


In [20]:
salary_change

,Region,Location_Context,SOC_Code,Skill_Category,Median_Salary_2023,Median_Salary_2025,Salary_Change_GBP_2023_2025,Salary_Change_Pct_2023_2025
0,London,London,2135,Cybersecurity,NaN,59590.0,NaN,NaN
1,North West,Manchester regional context,2135,Cybersecurity,59001.0,51954.0,-7047.0,-11.94
2,West Midlands,Birmingham regional context,2135,Cybersecurity,44505.0,NaN,NaN,NaN
3,London,London,3544,Data Analytics,40198.0,42850.0,2652.0,6.60
4,North West,Manchester regional context,3544,Data Analytics,31496.0,33871.0,2375.0,7.54
5,West Midlands,Birmingham regional context,3544,Data Analytics,31094.0,44172.0,13078.0,42.06
6,London,London,2134,Software Development,67505.0,74284.0,6779.0,10.04
7,North West,Manchester regional context,2134,Software Development,43207.0,49505.0,6298.0,14.58
8,West Midlands,Birmingham regional context,2134,Software Development,41315.0,47744.0,6429.0,15.56


In [21]:
import pandas as pd
from pathlib import Path

processed_path = Path(
    r"D:\MScProjects\labour-market-intelligence\data\processed"
)

model_df = pd.read_csv(
    processed_path / "integrated_digital_demand_salary_2023_2025.csv"
)

print("Shape:", model_df.shape)
print("\nColumns:")
print(model_df.columns.tolist())

print("\nMissing values:")
print(model_df.isna().sum())

model_df

Shape: (9, 10)

Columns:
['Year', 'SOC_Code', 'Occupation', 'Skill_Category', 'Observed_Months', 'Total_Job_Adverts', 'Average_Monthly_Adverts', 'YoY_Demand_Growth_Pct', 'Median_Annual_Salary', 'Salary_Growth_Pct']

Missing values:
Year                       0
SOC_Code                   0
Occupation                 0
Skill_Category             0
Observed_Months            0
Total_Job_Adverts          0
Average_Monthly_Adverts    0
YoY_Demand_Growth_Pct      3
Median_Annual_Salary       0
Salary_Growth_Pct          3
dtype: int64


,Year,SOC_Code,Occupation,Skill_Category,Observed_Months,Total_Job_Adverts,Average_Monthly_Adverts,YoY_Demand_Growth_Pct,Median_Annual_Salary,Salary_Growth_Pct
0,2023,2134,Programmers and software development professio...,Software Development,12,194205.0,16183.8,NaN,49444,NaN
1,2024,2134,Programmers and software development professio...,Software Development,12,98969.0,8247.4,-49.04,53363,7.93
2,2025,2134,Programmers and software development professio...,Software Development,12,113816.0,9484.7,15.00,55587,4.17
3,2023,2135,Cyber security professionals,Cybersecurity,12,29493.0,2457.8,NaN,45573,NaN
4,2024,2135,Cyber security professionals,Cybersecurity,12,17143.0,1428.6,-41.87,48555,6.54
5,2025,2135,Cyber security professionals,Cybersecurity,10,18460.0,1846.0,29.22,54816,12.89
6,2023,3544,Data analysts,Data Analytics,12,29599.0,2466.6,NaN,33695,NaN
7,2024,3544,Data analysts,Data Analytics,12,17516.0,1459.7,-40.82,34125,1.28
8,2025,3544,Data analysts,Data Analytics,12,19877.0,1656.4,13.48,38107,11.67


In [22]:
from pathlib import Path
import pandas as pd

processed_path = Path(
    r"D:\MScProjects\labour-market-intelligence\data\processed"
)

print("Processed CSV files:\n")

for f in processed_path.glob("*.csv"):
    print(f.name)

Processed CSV files:

city_salary_2023_2025.csv
dashboard_kpis.csv
demand_growth.csv
demand_ranking.csv
demand_salary_correlation.csv
digital_demand_2023_2025.csv
digital_demand_salary_summary_2023_2025.csv
digital_skill_classification.csv
integrated_digital_demand_salary_2023_2025.csv
master_dataset.csv
monthly_dashboard.csv
nlp_job_title_classification.csv
occupation_correlation.csv
occupation_dashboard.csv
occupation_summary.csv
occupation_summary_dashboard.csv
onet_selected_skills.csv
percentage_growth.csv
powerbi_digital_labour_market.csv
powerbi_master.csv
regional_digital_salary_2023_2025.csv
regional_salary_change_2023_2025.csv
regional_salary_premiums_2023_2025.csv
regression_results.csv
salary_2023_2025.csv
salary_ranking.csv
summary_statistics.csv
summary_statistics_master.csv
trend_dashboard.csv
uk_digital_job_demand_2023_2025.csv
yearly_average_demand.csv


In [23]:
files_to_check = [
    "digital_demand_2023_2025.csv",
    "monthly_dashboard.csv",
    "uk_digital_job_demand_2023_2025.csv"
]

for filename in files_to_check:
    path = processed_path / filename
    
    df_check = pd.read_csv(path)
    
    print("\n" + "=" * 70)
    print(filename)
    print("Shape:", df_check.shape)
    print("Columns:")
    print(df_check.columns.tolist())
    print("\nFirst 3 rows:")
    print(df_check.head(3).to_string(index=False))


digital_demand_2023_2025.csv
Shape: (29, 6)
Columns:
['Month', '2134 Programmers and software development professionals', '2135 Cyber security professionals', '2433 Actuaries, economists and statisticians', '3133 Database administrators and web content technicians', '3544 Data analysts']

First 3 rows:
     Month  2134 Programmers and software development professionals  2135 Cyber security professionals  2433 Actuaries, economists and statisticians  3133 Database administrators and web content technicians  3544 Data analysts
2023-01-01                                                  25269.0                             3712.0                                        3993.0                                                    2397.0              3606.0
2023-02-01                                                  25118.0                             3367.0                                        3131.0                                                    2084.0              3358.0
2023-03-01    

In [24]:
master = pd.read_csv(
    processed_path / "master_dataset.csv"
)

print("Shape:", master.shape)

print("\nColumns:")
print(master.columns.tolist())

print("\nFirst 10 rows:")
print(master.head(10).to_string(index=False))

print("\nMissing values:")
print(master.isna().sum())

# If Month exists, inspect coverage
if "Month" in master.columns:
    master["Month"] = pd.to_datetime(
        master["Month"],
        errors="coerce"
    )

    print("\nDate range:")
    print(master["Month"].min(), "to", master["Month"].max())

Shape: (145, 5)

Columns:
['Month', 'Occupation', 'Demand', 'SOC_Code', 'Median_Salary']

First 10 rows:
     Month                                         Occupation  Demand  SOC_Code  Median_Salary
2023-01-01 Programmers and software development professionals 25269.0    2134.0          52500
2023-02-01 Programmers and software development professionals 25118.0    2134.0          55000
2023-03-01 Programmers and software development professionals 21905.0    2134.0          55000
2023-04-01 Programmers and software development professionals 15409.0    2134.0          55000
2023-05-01 Programmers and software development professionals 17787.0    2134.0          55000
2023-06-01 Programmers and software development professionals 18059.0    2134.0          55000
2023-07-01 Programmers and software development professionals 14683.0    2134.0          55000
2023-08-01 Programmers and software development professionals 14362.0    2134.0          51600
2023-09-01 Programmers and software deve

In [25]:
import pandas as pd

dfs = {}

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        dfs[name] = obj.shape

print("DataFrames currently in memory:\n")

for name, shape in sorted(dfs.items()):
    print(f"{name:40s} {shape}")

DataFrames currently in memory:

_                                        (9, 10)
_20                                      (9, 8)
_21                                      (9, 10)
_4                                       (20, 20)
_6                                       (9, 6)
__                                       (9, 8)
___                                      (9, 6)
df_check                                 (9, 8)
master                                   (145, 5)
model_df                                 (9, 10)
region_2025                              (4537, 3)
region_2025_raw                          (4542, 20)
regional_analysis                        (27, 9)
regional_focus_2025                      (9, 6)
regional_premium_summary                 (18, 7)
regional_salary_all                      (27, 6)
salary_change                            (9, 8)
target_region_2025                       (33, 3)
wm_baseline                              (9, 3)


In [81]:
# Save monthly demand dataset for statistical modelling

monthly_model_path = (
    processed_path / "digital_monthly_demand_2023_2025.csv"
)

monthly_demand = demand_2325[
    [
        "Month",
        "Year",
        "SOC_Code",
        "Occupation",
        "Skill_Category",
        "Job_Adverts"
    ]
].copy()

monthly_demand = monthly_demand.sort_values(
    ["SOC_Code", "Month"]
).reset_index(drop=True)

monthly_demand.to_csv(
    monthly_model_path,
    index=False
)

print("Saved:", monthly_model_path)
print("Shape:", monthly_demand.shape)

print("\nDate range:")
print(
    monthly_demand["Month"].min(),
    "to",
    monthly_demand["Month"].max()
)

print("\nRows by occupation:")
print(
    monthly_demand.groupby(
        ["SOC_Code", "Skill_Category"]
    ).size()
)

print("\nMissing Job_Adverts:")
print(
    monthly_demand.groupby(
        ["SOC_Code", "Skill_Category"]
    )["Job_Adverts"].apply(lambda x: x.isna().sum())
)

Saved: D:\MScProjects\labour-market-intelligence\data\processed\digital_monthly_demand_2023_2025.csv
Shape: (108, 6)

Date range:
2023-01-01 00:00:00 to 2025-12-01 00:00:00

Rows by occupation:
SOC_Code  Skill_Category      
2134      Software Development    36
2135      Cybersecurity           36
3544      Data Analytics          36
dtype: int64

Missing Job_Adverts:
SOC_Code  Skill_Category      
2134      Software Development    0
2135      Cybersecurity           2
3544      Data Analytics          0
Name: Job_Adverts, dtype: int64
